# Compare GCP coordinate sets — Quinta Playa

Simplified: this only compares the **10 manually verified point correspondences** between SET_1 (`ground_control_points.txt`) and SET_2 (`GCP_2025.txt`) — the same 10 rows in `gcp_comparison_results.xlsx` used by the offset-correction notebook. No automatic nearest-neighbor matching, no SET_3 (handwritten logbook), no ambiguity/unmatched-point logic.

- **`known pair`** — correspondence confirmed by hand (station identity matched directly, not by proximity).
- **`auto match`** — correspondence originally found by nearest-neighbor search, since verified correct.

Elevation (dz) is reported for every pair but never used to decide anything — it's just the number that matters for the vertical shift.

**⚠️ 4 of the 10 points (CAMPAMENTO, 1B, 2B, C) don't have raw coordinates in this notebook yet** — I only had their already-computed dx/dy/dz offsets from the earlier comparison, not their source lat/lon/alt. Fill those in under SET_1 / SET_2 below (marked `# TODO`) to get this notebook computing all 10 rows itself; until then those 3 rows print/export with `None` and a flag instead of a fabricated number.

Final cell writes everything to `gcp_comparison_results.xlsx`. Requires `pandas` and `openpyxl` — run `%pip install pandas openpyxl` first if you don't have them.

In [ ]:
import math
import pandas as pd

REF_LAT_DEG = -1.005
M_PER_DEG_LAT = 111320.0
M_PER_DEG_LON = 111320.0 * math.cos(math.radians(REF_LAT_DEG))

EXCEL_OUTPUT_PATH = "gcp_comparison_results.xlsx"

## Set 1 — `ground_control_points.txt`

In [ ]:
# label -> (lon, lat, altitude_m)
SET_1 = {
    "PUNTO_CONTROL_CAMPAMENTO": None,  # TODO: fill in real (lon, lat, alt)
    "PUNTO_CONTROL_1B":         None,  # TODO: fill in real (lon, lat, alt)
    "PUNTO_CONTROL_2B":         None,  # TODO: fill in real (lon, lat, alt)
    "PUNTO_CONTROL_C":          None,  # TODO: fill in real (lon, lat, alt)
    "PUNTO_CONTROL_A1R":        (-91.087690353, -1.007849386, 2.0616),
    "PUNTO_CONTROL_B1R":        (-91.085474524, -1.006759953, 1.1852),
    "PUNTO_CONTROL_B2R":        (-91.082290413, -1.005638168, 1.4630),
    "PUNTO_CONTROL_B3R":        (-91.079938027, -1.005174261, 1.5965),
    "PUNTO_CONTROL_B4R":        (-91.074780876, -1.004605126, 2.1323),
    "PUNTO_CONTROL_C1R":        (-91.073226850, -1.004869312, 1.9402),
}
SET_1

## Set 2 — `GCP_2025.txt` (the file that gave the 8–9mm Metashape fit)

In [ ]:
SET_2 = {
    "BN_CAMPAMENTO_HITOCONTROL_QP": None,  # TODO: fill in real (lon, lat, alt)
    "BV_MEDIAB_HITOCONTROL_QP1":    None,  # TODO: fill in real (lon, lat, alt)
    "BV_B4_HITOCONTROL_QP6":        None,  # TODO: fill in real (lon, lat, alt)
    "BN_C1_HITOCONTROL_QP5":        None,  # TODO: fill in real (lon, lat, alt)
    "A1_HITOCONTROL_QP":            (-91.08768304, -1.00784055, 3.87360000),
    "B1_HITOCONTROL_QP":            (-91.08546709, -1.00675111, 3.01130000),
    "B2_HITOCONTROL_QP":            (-91.08228286, -1.00562935, 3.25120000),
    "B3_HITOCONTROL_QP2":           (-91.07993045, -1.00516543, 3.38860000),
    "C1_HITOCONTROL_QP4":           (-91.07321937, -1.00486026, 3.76150000),
    "B4_HITOCONTROL_QP7":           (-91.07477345, -1.00459609, 4.00260000),
}
SET_2

## Verified pairs

All 10 correspondences, in the same order as `gcp_comparison_results.xlsx`.

In [ ]:
VERIFIED_PAIRS = [
    ("PUNTO_CONTROL_CAMPAMENTO", "BN_CAMPAMENTO_HITOCONTROL_QP", "known pair"),
    ("PUNTO_CONTROL_1B",         "BV_MEDIAB_HITOCONTROL_QP1",    "known pair"),
    ("PUNTO_CONTROL_2B",         "BV_B4_HITOCONTROL_QP6",        "known pair"),
    ("PUNTO_CONTROL_C",          "BN_C1_HITOCONTROL_QP5",        "auto match"),
    ("PUNTO_CONTROL_A1R",        "A1_HITOCONTROL_QP",            "auto match"),
    ("PUNTO_CONTROL_B1R",        "B1_HITOCONTROL_QP",            "auto match"),
    ("PUNTO_CONTROL_B2R",        "B2_HITOCONTROL_QP",            "auto match"),
    ("PUNTO_CONTROL_B3R",        "B3_HITOCONTROL_QP2",           "auto match"),
    ("PUNTO_CONTROL_B4R",        "B4_HITOCONTROL_QP7",           "auto match"),
    ("PUNTO_CONTROL_C1R",        "C1_HITOCONTROL_QP4",           "auto match"),
]

## Compare and report

In [ ]:
def distance_components(p1, p2):
    """p1, p2 = (lon, lat, alt). Returns (horizontal_m, dx, dy, dz)."""
    lon1, lat1, alt1 = p1
    lon2, lat2, alt2 = p2
    dx = (lon2 - lon1) * M_PER_DEG_LON
    dy = (lat2 - lat1) * M_PER_DEG_LAT
    horiz = math.hypot(dx, dy)
    dz = None if (alt1 is None or alt2 is None) else (alt2 - alt1)
    return horiz, dx, dy, dz

def compare_verified_pairs(set_a, set_b, pairs):
    """Directly compares the manually-specified list of (label_a, label_b, row_type) pairs.
    A pair with a missing (None) coordinate on either side is reported with
    no computed values and a flag, instead of erroring out."""
    rows = []
    for label_a, label_b, row_type in pairs:
        point_a = set_a.get(label_a)
        point_b = set_b.get(label_b)
        if point_a is None or point_b is None:
            rows.append({
                "Row type": row_type,
                "Point A": label_a,
                "Point B": label_b,
                "Horizontal offset (m)": None,
                "dx (m)": None,
                "dy (m)": None,
                "Elevation offset dz (m)": None,
                "Flag": "MISSING COORDINATES -- fill in SET_1/SET_2 above",
            })
            continue
        horiz, dx, dy, dz = distance_components(point_a, point_b)
        rows.append({
            "Row type": row_type,
            "Point A": label_a,
            "Point B": label_b,
            "Horizontal offset (m)": round(horiz, 2),
            "dx (m)": round(dx, 2),
            "dy (m)": round(dy, 2),
            "Elevation offset dz (m)": round(dz, 2) if dz is not None else None,
            "Flag": "",
        })
    return rows

def print_report(rows):
    print(f"{'Row type':<12} {'Point A':<32} {'Point B':<32} {'horiz':>8} {'dx':>8} {'dy':>8} {'dz':>8}")
    for row in rows:
        if row["Flag"]:
            print(f"{row['Row type']:<12} {row['Point A']:<32} {row['Point B']:<32}  -- {row['Flag']}")
            continue
        print(f"{row['Row type']:<12} {row['Point A']:<32} {row['Point B']:<32} "
              f"{row['Horizontal offset (m)']:>7.2f}m {row['dx (m)']:>7.2f} {row['dy (m)']:>7.2f} {row['Elevation offset dz (m)']:>+7.2f}")

## Run and export

In [ ]:
rows = compare_verified_pairs(SET_1, SET_2, VERIFIED_PAIRS)
print_report(rows)

df = pd.DataFrame(rows)
df.to_excel(EXCEL_OUTPUT_PATH, index=False)
print(f"\nWrote {len(df)} rows to {EXCEL_OUTPUT_PATH}")
df